In [1]:
import numpy as np
from TDD.TDD import TDD, Ini_TDD,Clear_TDD,set_index_order,get_unique_table_num,set_root_of_unit,get_count,cont,renormalize,Slicing2,Slicing,renormalize_wio_op
from TDD.TDD_Q import cir_2_tn,get_real_qubit_num,add_trace_line,add_inputs,add_outputs,gen_cir
from TDD.TN import Index,Tensor,TensorNetwork
from TDD.Syn import *

import copy
from qiskit.circuit.library.standard_gates import HGate,U1Gate,U3Gate
from qiskit.quantum_info.operators import Operator
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Statevector
from qiskit.circuit import CircuitInstruction
import time
to_test = False

import warnings

warnings.simplefilter("ignore")

In [2]:
def oracle_syn1(tdd,data_q = -1,n_r=0):

    n = tdd.node.key+1
    cir = Circuit(n+1,[])
    if data_q==-1:
        data_q = n


    if tdd.neg:
        cir.data.append(Gate('x',{},data_q))
        
    if tdd.node.key==-1:
        if abs(tdd.weight)<1e-10:
            cir = Circuit(n_r+1,[])
            return cir
        else:
            cir.data.append(Gate('x',{},data_q))
        return cir

    u = tdd.node


    
    if tdd.node.successor[0]!=tdd.node.successor[1]:
        bran_tdd = get_branch_dd(u,0)
        cir_end_t0 = oracle_syn1(bran_tdd,data_q)        
        bran_tdd = get_branch_dd(u,1)
        cir_end_t1 = oracle_syn1(bran_tdd,data_q)
        cir_end_t0 = get_controlled_circuit2(cir_end_t0,{u.key:0},1,True)
        cir.data+=cir_end_t0.data
        
        reverse_q = []
        the_map = u.out_maps[1]
        while the_map.level>-1:
            q = the_map.level
            reverse_q.append(q)
            the_map=the_map.father
            
        cir_end_t1 = get_controlled_circuit2(cir_end_t1,{u.key:1},1,True)
        for g in cir_end_t1.data:
            for k in g.q_c:
                if k in reverse_q:
                    g.q_c[k] = 1-g.q_c[k]
        cir.data+=cir_end_t1.data
        
      
    else:       
        op_gates = []
        the_map = u.out_maps[1]
        while the_map.level>-1:
            q1 = u.key
            q = the_map.level
            g=Gate('x',{q1:1},q)
            g.is_edge_op = True
            op_gates.append(g)
            the_map=the_map.father
        cir.data+=op_gates


        bran_tdd = get_branch_dd(u,0,False)
        cir_end_t0 = oracle_syn1(bran_tdd,data_q)
        # print(cir_end_t0,'a')
        if abs(u.out_weight[0])>1e-10:
            if u.out_negs[0]:
                if abs(u.out_weight[1])>1e-10:
                    cir_end_t0.data=[Gate('x',{u.key:0},data_q)]+cir_end_t0.data
                else:
                    cir_end_t0.data=[Gate('x',{},data_q)]+cir_end_t0.data
            if u.out_negs[1]:
                cir_end_t0.data=[Gate('x',{u.key:1},data_q)]+cir_end_t0.data
                    
            if abs(u.out_weight[1])<1e-10:
                cir_end_t0 = get_controlled_circuit2(cir_end_t0,{u.key:0},1,True)
            cir.data=cir.data+cir_end_t0.data
        
        # if abs(u.out_weight[1])<1e-10:
        #     cir_end_t0 = get_controlled_circuit2(cir_end_t0,{u.key:0},1,True)
        # cir.data+=cir_end_t0.data
        
        op_gates.reverse()
        cir.data+=op_gates

    
    reverse_q = []
    the_map=tdd.map
    while the_map.level>-1:
        q = the_map.level
        reverse_q.append(q)
        the_map=the_map.father 
    for g in cir.data:
        for k in g.q_c:
            if k in reverse_q:
                g.q_c[k] = 1-g.q_c[k]
    
    return cir

def oracle_syn2(tdd,n=0,cond={},data_q = -1,qa_used=False,n_r=0):

    if n==0:
        n = tdd.node.key+1
    cir = Circuit(n+2,[])
    qa = n
    data_q = n+1
    
    if tdd.node.key==-1:
        if abs(tdd.weight)<1e-10:
            cir = Circuit(n_r+2,[])
            data_q=n_r+1
        else:
            if qa_used:
                cir.data.append(Gate('x',{qa:1},data_q))
            else:
                cir.data.append(Gate('x',{},data_q))
        if tdd.neg:
            if qa_used:
                cir.data.append(Gate('x',{qa:1},data_q))
            else:
                cir.data.append(Gate('x',{},data_q))
        return cir
    
    if tdd.node.key<n/2 and n>3:
        cir_t = oracle_syn1(tdd,data_q)
        cir_t = get_controlled_circuit2(cir_t,{qa:1},1,True)
        return cir_t

    
    if tdd.neg:
        if qa_used:
            cir.data.append(Gate('x',{qa:1},data_q))
        else:
            cir.data.append(Gate('x',{},data_q))

    u = tdd.node


    if u.successor[0]!=u.successor[1]:
        #把1分支关闭
        qa_used = True
        cond_n = copy.copy(cond)
        cond_n[u.key] = 1
        g = Gate('x',cond_n,qa)
        g.is_edge_op = True
        cir.data.append(g)

        bran_tdd = get_branch_dd(u,0)
        cir_end_t0 = oracle_syn2(bran_tdd,n,cond|{u.key:0},data_q,qa_used)
        cir.data=cir.data+cir_end_t0.data
    
        #把0分支关闭，1分支打开
        g=Gate('x',cond,qa)
        g.is_edge_op = True
        cir.data.append(g)

        reverse_q = []
        the_map = u.out_maps[1]
        while the_map.level>-1:
            q = the_map.level
            reverse_q.append(q)
            the_map=the_map.father
            
        bran_tdd = get_branch_dd(u,1)
        cir_end_t1 = oracle_syn2(bran_tdd,n,cond|{u.key:1},data_q,qa_used)
        for g in cir_end_t1.data:
            for k in g.q_c:
                if k in reverse_q:
                    g.q_c[k] = 1-g.q_c[k]        
        cir.data=cir.data+cir_end_t1.data

        #把0分支再打开
        cond_n = copy.copy(cond)
        cond_n[u.key] = 0
        g=Gate('x',cond_n,qa)
        g.is_edge_op = True
        cir.data.append(g)
    else:
        op_gates = []
        the_map = u.out_maps[1]
        while the_map.level>-1:
            q1 = u.key
            q = the_map.level
            g=Gate('x',{q1:1},q)
            g.is_edge_op = True
            op_gates.append(g)
            the_map=the_map.father
        cir.data+=op_gates
        bran_tdd = get_branch_dd(u,0,False)
        cir_end_t0 = oracle_syn2(bran_tdd,n,cond,data_q,qa_used)
        # print(cir_end_t0,'a')
        if abs(u.out_weight[0])>1e-10:
            if u.out_negs[0]:
                if qa_used:
                    if abs(u.out_weight[1])>1e-10:
                        cir_end_t0.data=[Gate('x',{qa:1,u.key:0},data_q)]+cir_end_t0.data
                    else:
                        cir_end_t0.data=[Gate('x',{qa:1},data_q)]+cir_end_t0.data
                else:
                    if abs(u.out_weight[1])>1e-10:
                        cir_end_t0.data=[Gate('x',{u.key:0},data_q)]+cir_end_t0.data
                    else:
                        cir_end_t0.data=[Gate('x',{},data_q)]+cir_end_t0.data
            if u.out_negs[1]:
                if qa_used:
                    cir_end_t0.data=[Gate('x',{qa:1,u.key:1},data_q)]+cir_end_t0.data
                else:
                    cir_end_t0.data=[Gate('x',{u.key:1},data_q)]+cir_end_t0.data
                    
            if abs(u.out_weight[1])<1e-10:
                cir_end_t0 = get_controlled_circuit2(cir_end_t0,{u.key:0},1,True)
            cir.data=cir.data+cir_end_t0.data

            
        op_gates.reverse()
        cir.data+=op_gates
            
    reverse_q = []
    the_map=tdd.map
    while the_map.level>-1:
        q = the_map.level
        reverse_q.append(q)
        the_map=the_map.father 
    for g in cir.data:
        for k in g.q_c:
            if k in reverse_q:
                g.q_c[k] = 1-g.q_c[k]
    return cir

def verify_success(cir,data,n):
    # print(data)
    for k in range(2**n):
        input = format(k & ((1 << n) - 1), f'0{n}b')
        current_state = {n-1-q : int(input[q]) for q in range(n)}
        current_state[n] = 1
        current_state[n+1] = 0
        current_state_s = copy.copy(current_state)
        # print(current_state_s)
        for g in cir.data:
            flag = True
            for q in g.q_c:
                if g.q_c[q]!=current_state[q]:
                    flag = False
                    break
            if flag:
                current_state[g.q_t]=1-current_state[g.q_t]
        # print(current_state)
        
        for q in range(len(current_state)-1):
            if current_state[q]!=current_state_s[q]:
                # print('Not Successful, see input:',input)
                return False             
        if current_state[n+1]!=data[k]:
            # print('Not Successful, see input:',input)
            return False
    # print('Successful')
    return True

In [3]:
t_cost_list =[0,0,7,16,24,62,80,200,216,272,392,448,504,560,616,672,728]

def calc_tcost(circ):
    cost = 0
    for gate in circ.data:
        cost += t_cost_list[len(list(gate.q_c.keys()))]
    return cost

def all_var_orders(num_input):
    import itertools
    orders = []
    for order in list(itertools.permutations(range(num_input))):
        orders.append([f'x{i}' for i in order])
    return orders

In [4]:
def get_data(n,save_data=True,p=1,cases=500,verify_succ=False):

    input_num = n

    lines = []
    t_start = time.time()
    with open(f'data/exorcism_rand_tcost_{input_num}.txt', 'r') as f:
        lines = {int(k): int(v)
                 for line in f
                 for k, v in (line.strip().split(':'),)}
        
    better, eqv, worse = 0, 0, 0
    all_orders = all_var_orders(input_num)

    our_data = {}
    num = 0
    for tt in lines:
        if num<(p-1)*cases:
            num+=1
            continue
        if num>=p*cases:
            break
        tcost_exor = lines[tt]
        tcost = 100000
        flag = True
        for order in all_orders:
            tdd,data = get_TDD_from_truth_table(input_num, tt, order,add_neg=True)
            circ = oracle_syn2(tdd,n_r=input_num)
            tcost = min(tcost, calc_tcost(circ))
            if verify_succ:
                if not verify_success(circ,data,input_num):
                    print(order)
                    flag = False
                    break      
            
        our_data[tt] = tcost
        if num%max((cases//100),1)==0 or num==len(lines)-1:
            print("N = ", num,',  ',time.time()-t_start,',   ',better)
        num+=1
            # print("TT = ", tt, tcost_exor,tcost)
        if flag==False:
            print('Something Wrong!!!')
            break
    
        if tcost > tcost_exor: worse += 1
        elif tcost == tcost_exor: eqv += 1
        else: better += 1

    print("****** Comparison of TDD-based Method Alg. 2 to Exorcism  ******")
    print(f"number of input variables: {input_num}")
    print(f"lower  t-cost in {better}({better / cases * 100}%) cases")
    print(f"higher t-cost in {worse}({worse / cases * 100}%) cases")
    print(f"same   t-cost in {eqv}({eqv / cases * 100}%) cases")
    print("****** ******  ******")
    if save_data:
        with open(f'our_rand_tcost_v4_{input_num}_s_{p}.txt', 'w') as f:
            for k, v in our_data.items():
                f.write(f'{k}:{v}\n')

get_data(7,True,p=17,cases=1000)

N =  16000 ,   37.93232345581055 ,    0
N =  16010 ,   823.3335793018341 ,    10
N =  16020 ,   1702.862689256668 ,    20
N =  16030 ,   2159.7106442451477 ,    30
N =  16040 ,   2584.656533718109 ,    40
N =  16050 ,   3195.3558254241943 ,    49
N =  16060 ,   4077.6098895072937 ,    58
N =  16070 ,   4471.321948051453 ,    68
N =  16080 ,   4945.208753108978 ,    78
N =  16090 ,   5426.125002622604 ,    88
N =  16100 ,   5915.069779872894 ,    98
N =  16110 ,   6405.166031360626 ,    107
N =  16120 ,   6898.312698364258 ,    117
N =  16130 ,   7388.981599807739 ,    127
N =  16140 ,   7877.256870269775 ,    137
N =  16150 ,   8374.05041050911 ,    146
N =  16160 ,   8866.524189472198 ,    156
N =  16170 ,   9355.102900028229 ,    166
N =  16180 ,   9844.0530667305 ,    176
N =  16190 ,   10334.722361564636 ,    186
N =  16200 ,   10825.79599237442 ,    195
N =  16210 ,   11314.9232006073 ,    205
N =  16220 ,   11796.167431116104 ,    215
N =  16230 ,   12283.05259346962 ,    225
N =